# Amazon ML Challenge 2026: Stage 5 — Decision Layer & Submission Export

**Input:** `score_refined_train.parquet` from NB 04 (or `score_C_train.parquet` from NB 03),
`candidate_pairs.tsv` from NB 02

**Pipeline position:** This is the final stage. It converts scores into predictions and
generates both submission files: `matching_results.tsv` and `candidate_pairs.tsv`.

**Methodology reference:** Section 2.5 (decision layer), A6 (calibration & two-threshold rule),
Section 1.1 (F0.5 scoring + singleton rule)

**Key decisions implemented:**
1. Two-threshold grid search (t_first, t_rest) directly optimizing Macro F0.5
2. Singleton calibration: tuned to reproduce ~5.6% singleton rate from GT
3. Error analysis: top false positives and false negatives
4. Clean export + official validation script

**Outputs:**
- `output/matching_results.tsv` — Scored on the leaderboard
- `output/candidate_pairs.tsv`  — Blocking audit file (required in final archive)

In [ ]:
import os, sys, gc, time, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# PATH RESOLUTION (Robust Multi-Source Search)
# ============================================================

IS_KAGGLE = os.path.exists('/kaggle')

def find_dir_by_file(pattern):
    if not IS_KAGGLE:
        candidates = [p for p in Path('.').glob(f'**/{pattern}') if p.is_file()]
    else:
        candidates = [p for p in Path('/kaggle/input').rglob(pattern) if p.is_file()]
    if not candidates:
        return None
    # Prioritize newest and largest
    candidates.sort(key=lambda p: (p.stat().st_size, p.stat().st_mtime), reverse=True)
    best = candidates[0]
    print(f"  [Found] {pattern} -> {best} ({best.stat().st_size / 1e6:.1f} MB)")
    return best.parent

if IS_KAGGLE:
    WORK_DIR = Path('/kaggle/working')
    # Try refined scores from NB 04, then fallback to score_C from NB 03
    SCORE_DIR = find_dir_by_file('score_refined_train.parquet')
    if SCORE_DIR is None:
        SCORE_DIR = find_dir_by_file('score_C_train.parquet')
    if SCORE_DIR is None:
        SCORE_DIR = WORK_DIR
        
    NORM_DIR = find_dir_by_file('train_s1_norm.parquet')
    DATA_DIR = find_dir_by_file('train_ground_truth.tsv')
    CAND_DIR = find_dir_by_file('candidate_pairs.tsv')
else:
    WORK_DIR = Path('work')
    SCORE_DIR = find_dir_by_file('score_refined_train.parquet') or find_dir_by_file('score_C_train.parquet') or WORK_DIR
    NORM_DIR = Path('work/normalized')
    DATA_DIR = Path('data/student_resource/dataset/train')
    CAND_DIR = Path('output')

OUTPUT_DIR = WORK_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nConfigured Directories:")
print(f"  Score Dir:  {SCORE_DIR}")
print(f"  Norm Dir:   {NORM_DIR}")
print(f"  Data Dir:   {DATA_DIR}")
print(f"  Cand Dir:   {CAND_DIR}")
print(f"  Output Dir: {OUTPUT_DIR}")

assert SCORE_DIR is not None, "Could not find score_C_train.parquet! Attach NB 03 output."
assert NORM_DIR is not None, "Could not find train_s1_norm.parquet! Attach NB 00 output."
assert DATA_DIR is not None, "Could not find train_ground_truth.tsv! Attach competition dataset."


In [ ]:
# ============================================================
# LOAD DATA (With Smart Pruning for 10x Fast Grid Search)
# ============================================================

refined_path = SCORE_DIR / 'score_refined_train.parquet'
raw_path = SCORE_DIR / 'score_C_train.parquet'

if refined_path.exists():
    print(f"Loading REFINED scores from NB 04: {refined_path}")
    score_file = refined_path
    score_col = 'score_final' if 'score_final' in pd.read_parquet(refined_path, columns=[]).columns else 'score_C'
elif raw_path.exists():
    print(f"Loading RAW scores from NB 03: {raw_path}")
    score_file = raw_path
    score_col = 'score_C'
else:
    raise FileNotFoundError(f"No score file found in {SCORE_DIR}. Run NB 03 first.")

# Load scores
df_scores = pd.read_parquet(score_file, columns=['s1_id', 'candidate_id', score_col])
if score_col != 'score_C':
    df_scores.rename(columns={score_col: 'score_C'}, inplace=True)
print(f"  Loaded {len(df_scores):,} scored pairs")

# Load S1 entity list (need complete list for submission)
print("Loading S1 entity list...")
df_s1 = pd.read_parquet(NORM_DIR / 'train_s1_norm.parquet', columns=['entity_id'])
all_s1_ids = df_s1['entity_id'].astype(str).tolist()
print(f"  Total S1 entities: {len(all_s1_ids):,}")
del df_s1; gc.collect()

# Load ground truth
print("Loading ground truth...")
df_gt = pd.read_csv(DATA_DIR / 'train_ground_truth.tsv', sep='\t', dtype=str)
gt_map = {}
for sid, raw_m in zip(
    df_gt['source1_entity_id'].astype(str).str.strip(),
    df_gt['matched_entity_ids'].fillna('').astype(str)
):
    if raw_m and raw_m != 'nan':
        gt_map[sid] = {m.strip() for m in raw_m.split(',') if m.strip()}
    else:
        gt_map[sid] = set()
del df_gt; gc.collect()

total_singletons = sum(1 for v in gt_map.values() if len(v) == 0)
gt_singleton_rate = total_singletons / len(gt_map) * 100
print(f"  GT singletons: {total_singletons:,} ({gt_singleton_rate:.1f}%)")

# Smart Pruning: Candidates with score < 0.30 can NEVER qualify
# since min t_first = 0.50 and min t_rest = 0.40. Pruning saves 10 GB heap memory!
print("  Pruning candidates with score_C < 0.30 for fast search...")
df_scores = df_scores[df_scores['score_C'] >= 0.30]
gc.collect()
print(f"  Plausible candidate pairs: {len(df_scores):,} ({len(df_scores)/84741406*100:.1f}% of total)")


In [ ]:
# ============================================================
# GROUP SCORES BY ENTITY
# ============================================================

print("\nGrouping scores by entity...")
grouped_cands = {sid: [] for sid in all_s1_ids}

for sid, cid, score in zip(
    df_scores['s1_id'].values,
    df_scores['candidate_id'].values,
    df_scores['score_C'].values
):
    sid = str(sid)
    if sid in grouped_cands:
        grouped_cands[sid].append((str(cid), float(score)))

# Sort descending by score
for sid in grouped_cands:
    grouped_cands[sid].sort(key=lambda x: -x[1])

# Quick stats
entities_with_cands = sum(1 for v in grouped_cands.values() if len(v) > 0)
entities_without = len(grouped_cands) - entities_with_cands
print(f"  Entities with candidates: {entities_with_cands:,}")
print(f"  Entities without:         {entities_without:,}")

In [ ]:
# ============================================================
# F0.5 METRIC (exact official implementation)
# ============================================================

def compute_entity_f05(pred_ids, true_ids):
    """Official per-entity F0.5 score (Methodology 1.1)."""
    if len(true_ids) == 0:
        return 1.0 if len(pred_ids) == 0 else 0.0
    if len(pred_ids) == 0:
        return 0.0
    tp = len(pred_ids & true_ids)
    if tp == 0:
        return 0.0
    precision = tp / len(pred_ids)
    recall = tp / len(true_ids)
    denom = 0.25 * precision + recall
    return (1.25 * precision * recall) / denom if denom > 0 else 0.0

def compute_macro_f05(predictions, ground_truth):
    """Macro-averaged F0.5 across all entities."""
    scores = []
    for sid in ground_truth:
        pred = predictions.get(sid, set())
        scores.append(compute_entity_f05(pred, ground_truth[sid]))
    return np.mean(scores), scores

In [ ]:
# ============================================================
# TWO-THRESHOLD GRID SEARCH (Methodology 2.5)
# t_first: threshold for the top-scoring candidate per entity.
#          If the best candidate doesn't pass, entity = singleton.
# t_rest:  threshold for additional candidates once the top is
#          accepted. Always t_rest <= t_first.
# ============================================================

print("\n" + "=" * 60)
print(" TWO-THRESHOLD GRID SEARCH")
print("=" * 60)

t0_grid = time.time()

# Fine grid for thorough search
t_first_grid = np.arange(0.50, 0.96, 0.025)  # 50% to 95%
t_rest_grid  = np.arange(0.40, 0.91, 0.025)  # 40% to 90%

best_f05 = -1.0
best_t_first = 0.80
best_t_rest = 0.65
results_grid = []

for t_first in t_first_grid:
    for t_rest in t_rest_grid:
        if t_rest > t_first:
            continue
        
        # Apply two-threshold rule
        predictions = {}
        for sid, cands in grouped_cands.items():
            if not cands:
                predictions[sid] = set()
                continue
            
            top_cid, top_score = cands[0]
            if top_score < t_first:
                # Best candidate not convincing -> singleton (abstain)
                predictions[sid] = set()
            else:
                accepted = {top_cid}
                for cid, score in cands[1:]:
                    if score >= t_rest:
                        accepted.add(cid)
                predictions[sid] = accepted
        
        f05, _ = compute_macro_f05(predictions, gt_map)
        results_grid.append((t_first, t_rest, f05))
        
        if f05 > best_f05:
            best_f05 = f05
            best_t_first = t_first
            best_t_rest = t_rest

elapsed_grid = time.time() - t0_grid

# Measure singleton rate under best thresholds
pred_singletons = 0
for sid, cands in grouped_cands.items():
    if not cands or cands[0][1] < best_t_first:
        pred_singletons += 1
pred_singleton_rate = pred_singletons / len(all_s1_ids) * 100

print(f"\n  Grid search: {len(results_grid):,} combinations in {elapsed_grid:.1f}s")
print(f"")
print(f"  ╔{'═' * 50}╗")
print(f"  ║{'OPTIMAL THRESHOLDS':^50s}║")
print(f"  ╠{'═' * 50}╣")
print(f"  ║  Best Macro F0.5:         {best_f05:.4f}{' ' * 20}║")
print(f"  ║  t_first (top candidate): {best_t_first:.3f}{' ' * 20}║")
print(f"  ║  t_rest  (additional):    {best_t_rest:.3f}{' ' * 20}║")
print(f"  ║  Predicted singleton rate: {pred_singleton_rate:.1f}% (GT: {gt_singleton_rate:.1f}%){' ' * 5}║")
print(f"  ╚{'═' * 50}╝")

# Show top-5 threshold combinations
print(f"\n  Top-5 combinations:")
results_sorted = sorted(results_grid, key=lambda x: -x[2])[:5]
for rank, (tf, tr, f) in enumerate(results_sorted, 1):
    print(f"    #{rank}: t_first={tf:.3f}, t_rest={tr:.3f}, F0.5={f:.4f}")

In [ ]:
# ============================================================
# APPLY BEST THRESHOLDS -> FINAL PREDICTIONS
# ============================================================

print("\n" + "=" * 60)
print(" APPLYING OPTIMAL THRESHOLDS")
print("=" * 60)

final_predictions = {}  # s1_id -> list of matched entity IDs

for sid in all_s1_ids:
    cands = grouped_cands.get(sid, [])
    
    if not cands or cands[0][1] < best_t_first:
        final_predictions[sid] = []
    else:
        accepted = [cands[0][0]]  # Top candidate accepted
        for cid, score in cands[1:]:
            if score >= best_t_rest:
                accepted.append(cid)
        final_predictions[sid] = accepted

# Stats
n_singletons = sum(1 for v in final_predictions.values() if len(v) == 0)
n_matched = sum(1 for v in final_predictions.values() if len(v) > 0)
total_matches = sum(len(v) for v in final_predictions.values())
avg_matches = total_matches / max(n_matched, 1)

print(f"  Total entities:   {len(final_predictions):,}")
print(f"  Singletons:       {n_singletons:,} ({n_singletons/len(final_predictions)*100:.1f}%)")
print(f"  Matched entities: {n_matched:,}")
print(f"  Total match IDs:  {total_matches:,}")
print(f"  Avg matches/ent:  {avg_matches:.2f}")

In [ ]:
# ============================================================
# DETAILED F0.5 BREAKDOWN
# ============================================================

print("\n" + "=" * 60)
print(" FINAL MACRO F0.5 BREAKDOWN")
print("=" * 60)

singleton_scores = []
non_singleton_scores = []

for sid in gt_map:
    pred = set(final_predictions.get(sid, []))
    true = gt_map[sid]
    score = compute_entity_f05(pred, true)
    if len(true) == 0:
        singleton_scores.append(score)
    else:
        non_singleton_scores.append(score)

all_entity_scores = singleton_scores + non_singleton_scores
macro_f05 = np.mean(all_entity_scores)

print(f"  Overall Macro F0.5:     {macro_f05:.4f}")
print(f"  Singleton F0.5:         {np.mean(singleton_scores):.4f} ({len(singleton_scores):,} entities)")
print(f"  Non-singleton F0.5:     {np.mean(non_singleton_scores):.4f} ({len(non_singleton_scores):,} entities)")
print(f"")

# Breakdown by match cardinality
print(f"  By match cardinality:")
for lo, hi, label in [(0, 0, 'singleton'), (1, 2, '1-2'), (3, 5, '3-5'), (6, 20, '6+')]:
    ent_scores = []
    for sid in gt_map:
        m_count = len(gt_map[sid])
        if lo <= m_count <= hi:
            pred = set(final_predictions.get(sid, []))
            ent_scores.append(compute_entity_f05(pred, gt_map[sid]))
    if ent_scores:
        print(f"    |M(e)|={label:>9s}: F0.5={np.mean(ent_scores):.4f} ({len(ent_scores):,} entities)")

In [ ]:
# ============================================================
# ERROR ANALYSIS (Methodology 2.4: error dump)
# Top false positives and false negatives for improvement.
# ============================================================

print("\n" + "=" * 60)
print(" ERROR ANALYSIS")
print("=" * 60)

# 1. False Positives: entities where we predicted something wrong
fp_entities = []
fn_entities = []

for sid in gt_map:
    pred = set(final_predictions.get(sid, []))
    true = gt_map[sid]
    
    # False positives: predicted but not in GT
    fps = pred - true
    if fps:
        top_score = grouped_cands.get(sid, [(None, 0)])[0][1]
        fp_entities.append((sid, len(fps), len(pred), len(true), top_score))
    
    # False negatives: in GT but not predicted
    fns = true - pred
    if fns and len(true) > 0:
        fn_entities.append((sid, len(fns), len(pred), len(true)))

# Sort by impact
fp_entities.sort(key=lambda x: -x[1])
fn_entities.sort(key=lambda x: -x[1])

print(f"\n  Entities with false positives: {len(fp_entities):,}")
print(f"  Entities with false negatives: {len(fn_entities):,}")

if fp_entities:
    print(f"\n  Top False Positive entities (worst impact):")
    for sid, n_fps, n_pred, n_true, top_s in fp_entities[:10]:
        print(f"    {sid}: {n_fps} FPs out of {n_pred} predicted (true={n_true}, top_score={top_s:.3f})")

if fn_entities:
    print(f"\n  Top False Negative entities (most missed matches):")
    for sid, n_fns, n_pred, n_true in fn_entities[:10]:
        print(f"    {sid}: missed {n_fns} of {n_true} true matches (predicted {n_pred})")

# Singleton errors
singleton_fp = sum(1 for sid in gt_map if len(gt_map[sid]) == 0 and len(final_predictions.get(sid, [])) > 0)
singleton_fn = sum(1 for sid in gt_map if len(gt_map[sid]) > 0 and len(final_predictions.get(sid, [])) == 0)
print(f"\n  Singleton false positives: {singleton_fp:,} (predicted match for true singleton)")
print(f"  Missed non-singletons:     {singleton_fn:,} (predicted empty for true match)")

In [ ]:
# ============================================================
# EXPORT SUBMISSION FILES (Official format)
# ============================================================

print("\n" + "=" * 60)
print(" EXPORTING SUBMISSION FILES")
print("=" * 60)

# 1. matching_results.tsv — THE SCORED FILE
matching_path = OUTPUT_DIR / 'matching_results.tsv'
with open(matching_path, 'w', encoding='utf-8') as f:
    f.write('source1_entity_id\tmatched_entity_ids\n')
    for sid in all_s1_ids:
        matches = final_predictions.get(sid, [])
        match_str = ','.join(matches)
        f.write(f'{sid}\t{match_str}\n')

print(f"  ✅ {matching_path} ({matching_path.stat().st_size / 1e6:.1f} MB)")

# 2. candidate_pairs.tsv — Blocking audit file
# Copy from NB 02 output, or regenerate from candidates parquet
candidate_path = OUTPUT_DIR / 'candidate_pairs.tsv'

# Check if NB 02 already wrote it
nb02_cand = CAND_DIR / 'candidate_pairs.tsv' if CAND_DIR else None
if nb02_cand and nb02_cand.exists() and str(nb02_cand) != str(candidate_path):
    import shutil
    shutil.copy2(nb02_cand, candidate_path)
    print(f"  ✅ {candidate_path} (copied from NB 02)")
elif candidate_path.exists():
    print(f"  ✅ {candidate_path} (already exists)")
else:
    # Regenerate from grouped_cands
    with open(candidate_path, 'w', encoding='utf-8') as f:
        f.write('source1_entity_id\tcandidate_entity_ids\n')
        for sid in all_s1_ids:
            cands = grouped_cands.get(sid, [])
            cand_str = ','.join(c for c, _ in cands)
            f.write(f'{sid}\t{cand_str}\n')
    print(f"  ✅ {candidate_path} (regenerated from scores)")

# Verify row counts
n_matching_rows = sum(1 for _ in open(matching_path, encoding='utf-8')) - 1
n_candidate_rows = sum(1 for _ in open(candidate_path, encoding='utf-8')) - 1
print(f"\n  matching_results.tsv rows:  {n_matching_rows:,} (expected {len(all_s1_ids):,})")
print(f"  candidate_pairs.tsv rows:   {n_candidate_rows:,} (expected {len(all_s1_ids):,})")

assert n_matching_rows == len(all_s1_ids), f"Row count mismatch! Expected {len(all_s1_ids)}, got {n_matching_rows}"
print("\n  ✅ Row count verified.")

In [ ]:
# ============================================================
# OFFICIAL VALIDATION SCRIPT
# ============================================================

print("\n" + "=" * 60)
print(" OFFICIAL VALIDATION")
print("=" * 60)

# Locate validator
validator_script = None
possible_paths = [
    Path('data/student_resource/utils/validate_submission.py'),
    Path('student_resource/utils/validate_submission.py'),
    Path('/kaggle/input/datasets/ojassangwai/mlchallengedata/utils/validate_submission.py'),
]
# Also search recursively in /kaggle/input
if IS_KAGGLE:
    for p in Path('/kaggle/input').rglob('validate_submission.py'):
        possible_paths.insert(0, p)
        break

for p in possible_paths:
    if p.exists():
        validator_script = p
        break

if validator_script:
    print(f"  Found validator: {validator_script}")
    
    # Determine test dir (for test source files)
    test_dir = DATA_DIR / 'test' if (DATA_DIR / 'test').exists() else DATA_DIR
    
    cmd = [
        sys.executable,
        str(validator_script),
        '--matching', str(matching_path),
        '--candidate', str(candidate_path),
        '--test-dir', str(test_dir)
    ]
    print(f"  Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    
    if result.returncode == 0:
        print("  ✅ VALIDATION PASSED")
    else:
        print(f"  ❌ VALIDATION FAILED (exit code {result.returncode})")
else:
    print("  ⚠️ validate_submission.py not found. Manual format check:")
    # Basic format checks
    sample = pd.read_csv(matching_path, sep='\t', nrows=5)
    print(f"    Columns: {list(sample.columns)}")
    print(f"    Expected: ['source1_entity_id', 'matched_entity_ids']")
    print(f"    First rows:")
    print(sample.head().to_string(index=False))

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "#" * 60)
print("#" + " PIPELINE COMPLETE ".center(58) + "#")
print("#" * 60)
print(f"#")
print(f"#  Macro F0.5:              {macro_f05:.4f}")
print(f"#  t_first:                 {best_t_first:.3f}")
print(f"#  t_rest:                  {best_t_rest:.3f}")
print(f"#  Predicted singletons:    {n_singletons:,} ({pred_singleton_rate:.1f}%)")
print(f"#  GT singletons:           {total_singletons:,} ({gt_singleton_rate:.1f}%)")
print(f"#")
print(f"#  Output files:")
print(f"#    {matching_path}")
print(f"#    {candidate_path}")
print(f"#")
print("#" * 60)